# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/T0othIess/FlyRank-AI-ML-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

Ranking, because specifying which page needs to reviewed first for CTR improvement is the definition of ranking,
aka ordering thing from most to least important

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

i'd pick proxy, target isnt possible as it wont be accurate, if i pick CTR as the as the target, pages differ in CTR depending on position so its not a clear prediction.
the proxy i'd pick is CTR difference (expected CTR - actual CTR), this would give me better accuracy.

In [12]:
import pandas as pd
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")
df["expected_ctr_per_tier"] = df.groupby("position_tier")["ctr"].transform("mean") # so like transform keeps the rows as is, when like .mean() would just give the mean for the piles grouby gives.
df["ctr_gap"] = df["expected_ctr_per_tier"] - df["ctr"]
df[["position_tier","expected_ctr_per_tier", "ctr", "ctr_gap"]].head(10).style.format("{:.3f}",subset=["expected_ctr_per_tier", "ctr", "ctr_gap"]).set_properties(**{"text-align": "center"})

,position_tier,expected_ctr_per_tier,ctr,ctr_gap
0,striking,0.323,0.760,-0.437
1,page_3_5,0.222,0.050,0.172
2,page_3_5,0.222,0.090,0.132
3,page_1,0.652,0.490,0.162
4,page_3_5,0.222,0.130,0.092
5,page_1,0.652,0.030,0.622
6,page_1,0.652,0.000,0.652
7,page_3_5,0.222,0.060,0.162
8,page_3_5,0.222,0.090,0.132
9,page_1,0.652,0.160,0.492


Note: if its negative that means it outperforms what was expected

## 3. Success metric

*One metric you can defend. What number means 'good'?*

Success metric: Precision@K — of my top K ranked pages (by ctr_gap), what fraction also have real engagement (sessions_90d > 10)? Sessions weren't used in the ranking, so this checks something independent.  
Result: Precision@20 = 0.30, Precision@50 = 0.34.

What "good" means: I'd want this above ~0.5 — right now my ranking often surfaces high-visibility pages with little real traffic, suggesting impressions is a weak tiebreaker.

In [ ]:
# note: the reason im filtering is because top_3 positive has unrealistic expected_ctr (>1)
ranked_filtered_df = df.query("expected_ctr_per_tier <1 and impressions_90d >1000").sort_values(["ctr_gap", "impressions_90d"], ascending=[False, False])
#note for above: impressions_90d works a tiebreaker, priority is ctr_gap with sorting, but when there is a tie then it'll sort based on impressions_90d

top_50 = ranked_filtered_df.head(50)
opportunity_at_50 = top_50.query("sessions_90d > 10")

top_20 = ranked_filtered_df.head(20)
opportunity_at_20 = top_20.query("sessions_90d > 10")
precision_50 = len(opportunity_at_50) / len(top_50)
precision_20 = len(opportunity_at_20) / len(top_20)
print(f"Precision@20: {precision_20}")
print(f"Precision@50: {precision_50}")

Precision@20: 0.3
Precision@50: 0.34


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*
one row is equal to one page with its own unique content id, CTR, position tier, impressions and my OWN row CTR gap


In [56]:
df[["content_id", "position_tier", "impressions_90d","sessions_90d", "expected_ctr_per_tier", "ctr", "ctr_gap"]].head(10) \
    .style.format("{:.3f}", subset=["ctr", "expected_ctr_per_tier", "ctr_gap"]).set_properties(**{"text-align": "center"})

,content_id,position_tier,impressions_90d,sessions_90d,expected_ctr_per_tier,ctr,ctr_gap
0,content_304f48230142,striking,3803,17,0.323,0.760,-0.437
1,content_a1fb4e703a9e,page_3_5,15320,9,0.222,0.050,0.172
2,content_9aa793d4d895,page_3_5,12581,11,0.222,0.090,0.132
3,content_331d6c4de07b,page_1,11751,78,0.652,0.490,0.162
4,content_d99b7a2d90ca,page_3_5,19140,145,0.222,0.130,0.092
5,content_d4084a4bc775,page_1,3970,5,0.652,0.030,0.622
6,content_9a34b442b552,page_1,20,1,0.652,0.000,0.652
7,content_a63219c6e95a,page_3_5,1724,28,0.222,0.060,0.162
8,content_5e6c160719bc,page_3_5,32574,68,0.222,0.090,0.132
9,content_c27558df2b0c,page_1,1240,3,0.652,0.160,0.492


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

my rule (ctr_gap = expected_ctr - actual ctr) only uses 2 variables and the results showed like 0.3 precision, which means they arent enough.  
a model would be better at this because it could include more factors into this instead of me trying to guess which one matters the most.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ x ] Every section above is filled — markdown thinking AND the code that backs it
- [ x ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ x ] No client names, URLs, or private queries anywhere
- [ x ] My claims use careful words: observed, measured, directional, decision-support
- [ x ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.